# Linear Encoding Image ranking



1. Start with the existing Net2Brain encoding results
2. Determine which layer to use
3. PCA dimensionality reduction: 1000×1280→1000×100  
4. Three-fold cross-validation: split train/test. out-of-sample prediction for every image.
5. Train Ridge regression Y=XW+b This is essentially many linear regressions on the training split.
6. Predict the held-out images
7. Calculate one MSE per image
8. Repeat independently for both models
9. Rank the 872 images
10. Spearman's rank correlation



Y: brain voxels. Y.shape = (872, voxels in roi)  
X: model features. ex: X.shape = (872, 1024)  

Selects two models.  
Selects the feature layer for each model.  
Loads the model features.   
Loads the fMRI ROI data.  
Runs 3-fold cross-validated PCA + Ridge.  
Gets one out-of-sample MSE per image.  
Converts MSEs into ranks.  
Calculates the Spearman correlation between the two model rankings.  
Saves the imagewise CSV.  
Saves the top-20 images for each model.    
Saves the biggest disagreements.  
Saves a subject-level summary.    

### figuring out best layer

In [ ]:
#inspect
import os
import glob
import pandas as pd
# ============================================================
# INSPECT AN ENCODING CSV
# ============================================================

encoding_root = "PATH"

ENCODING_FOLDERS = {
    "MAE": "PATH",
    "DINOv1": "PATH",
    "LEJEPA_vitl": "PATH",
    "IBOT": "PATH",
    "LEJEPA_vitb":"PATH",
    
}

model_to_inspect = 'MAE'
subject_to_inspect = "subj01"

folder = ENCODING_FOLDERS[model_to_inspect]

path = os.path.join(
    encoding_root,
    folder,
    subject_to_inspect,
    "*.csv"
)

csv_files = glob.glob(path)

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No encoding CSV found for {model_to_inspect} {subject_to_inspect}"
    )

df = pd.read_csv(csv_files[0])

print(df.shape)
print(df.columns.tolist())
print(df.head(20).to_string(index=False))

In [ ]:
# calc best layer per roi

import os
import glob
import pandas as pd

# ============================================================
# PATHS
# ============================================================

encoding_root = "PATH"

output_root = "PATH" ######

os.makedirs(output_root, exist_ok=True)

full_results_dir = os.path.join(output_root, "encoding_results")
summary_dir = os.path.join(output_root, "summary")

os.makedirs(full_results_dir, exist_ok=True)
os.makedirs(summary_dir, exist_ok=True)


# ============================================================
# CHOOSE MODELS
# ============================================================

MODEL1 = "VJEPA" ########
MODEL2 = "LeVJEPA" ########

ENCODING_FOLDERS = {
    "DINOv1": "PATH",
    "DINOv2": "PATH",

    "IBOT": "PATH",

    "INFONCE": "PATH",

    "LeJEPA": "PATH",
    "LeJEPA-B": "PATH",

    "LeVJEPA": "PATH",

    "MAE": "PATH",

    "OPENCLIP": "PATH",

    "VideoMAE": "PATH",

    "VJEPA": "PATH",

    "LeVLJEPA": "PATH",
}

models = {
    MODEL1: ENCODING_FOLDERS[MODEL1],
    MODEL2: ENCODING_FOLDERS[MODEL2],
}


# ============================================================
# READ ALL EXISTING ENCODING RESULTS
# ============================================================

all_results = []

for model_name, folder_name in models.items():

    model_dir = os.path.join(encoding_root, folder_name)

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    for subject in range(1, 9):

        current_subj = f"subj{subject:02d}"
        subj_dir = os.path.join(model_dir, current_subj)

        csv_files = glob.glob(os.path.join(subj_dir, "*.csv"))

        if len(csv_files) == 0:
            print(f"WARNING: No CSV found for {model_name} {current_subj}")
            continue

        if len(csv_files) > 1:
            print(f"WARNING: Multiple CSVs found for {model_name} {current_subj}:")
            for f in csv_files:
                print("   ", os.path.basename(f))

        # Use the first CSV
        csv_path = csv_files[0]

        df = pd.read_csv(csv_path)

        # Keep only the information we need
        df["Subject"] = current_subj
        df["Model"] = model_name

        all_results.append(df)

        print(
            f"{current_subj}: {len(df)} rows | "
            f"{df['ROI'].nunique()} ROIs | "
            f"{df['Laye"PATH"
        )


# ============================================================
# COMBINE EVERYTHING
# ============================================================

all_encoding = pd.concat(all_results, ignore_index=True)

# Save complete layer-wise results
all_encoding_path = os.path.join(
    full_results_dir,
    "all_layer_encoding_results.csv"
)

all_encoding.to_csv(all_encoding_path, index=False)

print("\n" + "=" * 70)
print("RESULTS")
print("=" * 70)

print("Rows:", len(all_encoding))
print("Models:", all_encoding["Model"].unique())
print("Subjects:", all_encoding["Subject"].nunique())
print("ROIs:", all_encoding["ROI"].nunique())
print("Layers:", all_encoding["Layer"].nunique())

print("\nSaved:")
print(all_encoding_path)


# ============================================================
# FIND BEST LAYER
# ============================================================
#
# For each:
#   Model × Subject × ROI
#
# choose the layer with the highest R.
#
# This uses the R already calculated by Net2Brain.
# ============================================================

best_rows = []

group_columns = ["Model", "Subject", "ROI"]

for group_values, group in all_encoding.groupby(group_columns):

    # Remove rows where R is missing
    group = group.dropna(subset=["R"])

    if len(group) == 0:
        continue

    # Highest encoding performance
    best_idx = group["R"].idxmax()

    best = group.loc[best_idx]

    best_rows.append({
        "Model": best["Model"],
        "Subject": best["Subject"],
        "ROI": best["ROI"],
        "Best_Laye"PATH"],
        "Best_"PATH"],
        "Significance": best["Significance"],
        "SEM": best["SEM"],
    })


best_layers = pd.DataFrame(best_rows)

# Sort nicely
best_layers = best_layers.sort_values(
    ["Model", "Subject", "ROI"]
).reset_index(drop=True)


# ============================================================
# SAVE BEST LAYERS
# ============================================================

best_layers_path = os.path.join(
    summary_dir,
    "best_layer_by_subject_roi.csv"
)

best_layers.to_csv(best_layers_path, index=False)

print("\n" + "=" * 70)
print("BEST LAYER PER SUBJECT × ROI")
print("=" * 70)

print(best_layers.head(20).to_string(index=False))

print("\nSaved:")
print(best_layers_path)


# ============================================================
# OVERALL BEST LAYER
# ============================================================
#
# Calculate the mean R for each layer across:
#   subjects × ROIs
#
# separately for MAE and DINO.
#
# This gives us a global layer preference for each model.
# ============================================================

overall_layer = (
    all_encoding
    .groupby(["Model", "Layer"])
    .agg(
        Mean_R=(""PATH"),
        Median_R=(""PATH"),
        Std_R=(""PATH"),
        N=(""PATH")
    )
    .reset_index()
)

# Rank layers within each model
overall_layer["Layer_Rank"] = (
    overall_layer
    .groupby("Model")["Mean_R"]
    .rank(method="min", ascending=False)
)

overall_layer = overall_layer.sort_values(
    ["Model", "Layer_Rank"]
).reset_index(drop=True)


overall_layer_path = os.path.join(
    summary_dir,
    "overall_layer_performance.csv"
)

overall_layer.to_csv(overall_layer_path, index=False)

print("\n" + "=" * 70)
print("OVERALL LAYER PERFORMANCE")
print("=" * 70)

print(overall_layer.to_string(index=False))

print("\nSaved:")
print(overall_layer_path)


# ============================================================
# GLOBAL BEST LAYER PER MODEL
# ============================================================

global_best = (
    overall_layer
    .sort_values(["Model", "Mean_R"], ascending=[True, False])
    .groupby("Model", as_index=False)
    .first()
)

global_best_path = os.path.join(
    summary_dir,
    "global_best_layer.csv"
)

global_best.to_csv(global_best_path, index=False)


# ============================================================
# BEST-LAYER COUNTS
# ============================================================
#
# How often was each layer selected as the best layer?
# This is useful for seeing whether a model consistently
# prefers certain depths.
# ============================================================

layer_counts = (
    best_layers
    .groupby(["Model", "Best_Layer"])
    .size()
    .reset_index(name="Number_of_ROIs")
)

layer_counts["Percentage"] = (
    layer_counts["Number_of_ROIs"]
    / best_layers.groupby("Model")["ROI"].count()
    .reindex(layer_counts["Model"])
    .values
    * 100
)

layer_counts = layer_counts.sort_values(
    ["Model", "Number_of_ROIs"],
    ascending=[True, False]
)

layer_counts_path = os.path.join(
    summary_dir,
    "best_layer_counts.csv"
)

layer_counts.to_csv(layer_counts_path, index=False)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("GLOBAL BEST LAYERS")
print("=" * 70)

print(
    global_best[
        ["Model", "Laye"PATH"]
    ].to_string(index=False)
)

print("\n" + "=" * 70)
print("HOW OFTEN EACH LAYER WAS BEST")
print("=" * 70)

print(layer_counts.to_string(index=False))

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

print(f"""
Output folder:

{output_root}

Files:

1. encoding_results/
   all_layer_encoding_results.csv

2. summary/
   best_layer_by_subject_roi.csv
   overall_layer_performance.csv
   global_best_layer.csv
   best_layer_counts.csv
""")

## run IMAGEWISE RIDGE ANALYSIS

In [ ]:

# ============================================================
# IMAGEWISE RIDGE ANALYSIS
# Best layer per model × subject × ROI
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from scipy.stats import spearmanr

# ============================================================
# CHOOSE THE TWO MODELS HERE
# ============================================================
######################################
MODEL1 = "VJEPA"
MODEL2 = "LeVJEPA"
#####################################
# ============================================================
# MODEL CONFIGURATION
# ============================================================

MODEL_CONFIG = {
    "DINOv1": {
        "feat_path": "PATH",
    },

    "DINOv2": {
        "feat_path": "PATH",
    },

    "IBOT": {
        "feat_path": "PATH",
    },

    "INFONCE": {
        "feat_path": "PATH",
    },

    "LeJEPA-B": {
        "feat_path": "PATH",
    },

    "LeJEPA": {
        "feat_path": "PATH",
    },

    "MAE": {
        "feat_path": "PATH",
    },

    "OPENCLIP": {
        "feat_path": "PATH",
    },
    "VideoMAE": {
        "feat_path": "PATH"
    },
    "VJEPA": {
        "feat_path": "PATH"
    },
    "LeVJEPA": {
        "feat_path": "PATH"
    },
    "LeVLJEPA": {
        "feat_path": "PATH"
    }
}

# ============================================================
# BEST-LAYER TABLE
# ============================================================
#############################################################
BEST_LAYER_PATH = ""
#############################################################
best_layers = pd.read_csv(BEST_LAYER_PATH)

# ============================================================
# CHECK SELECTED MODELS
# ============================================================

if MODEL1 not in MODEL_CONFIG or MODEL2 not in MODEL_CONFIG:
    raise ValueError("One of the selected models is not in MODEL_CONFIG.")

# ============================================================
# PATHS
# ============================================================

roi_root = "PATH"

output_root = os.path.join(
    "PATH",
    f"{MODEL1}-{MODEL2} best layer"
)

os.makedirs(output_root, exist_ok=True)

# ============================================================
# CROSS-VALIDATION SETTINGS
# ============================================================

n_folds = 3
n_components = 100
alpha = 1.0
random_state = 42

# ============================================================
# CREATE OUTPUT DIRECTORIES
# ============================================================

for model_name in [MODEL1, MODEL2]:
    os.makedirs(os.path.join(output_root, model_name), exist_ok=True)

# ============================================================
# GET FEATURE FILES
# ============================================================

def get_feature_files(feat_path):
    feat_files = glob.glob(os.path.join(feat_path, "*.np[zy]"))
    feat_files.sort()
    return feat_files

# ============================================================
# LOAD FEATURES FROM ONE LAYER
# ============================================================

def load_layer_features(feat_files, layer_id):
    X = []
    for file in feat_files:
        feat = np.load(file, allow_pickle=True)
        activation = feat[layer_id]
        activation = activation.flatten()
        X.append(activation)
    return np.array(X)

# ============================================================
# IMAGEWISE RIDGE REGRESSION
# ============================================================

def run_imagewise_ridge(X, Y, n_folds=3, n_components=100, alpha=1.0, random_state=42):
    kfold = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    image_mse = np.zeros(len(X))
    image_seen = np.zeros(len(X), dtype=bool)

    for fold, (train_idx, test_idx) in enumerate(kfold.split(X), 1):
        print(f"Fold {fold}: {len(train_idx)} training, {len(test_idx)} testing")

        X_train = X[train_idx]
        X_test = X[test_idx]
        Y_train = Y[train_idx]
        Y_test = Y[test_idx]

        pca = PCA(n_components=n_components, random_state=random_state)

        X_train_pca = pca.fit_transform(X_train)
        X_test_pca = pca.transform(X_test)

        print(f"  PCA: {X_train_pca.shape} -> {X_test_pca.shape}")

        reg = Ridge(alpha=alpha)
        reg.fit(X_train_pca, Y_train)

        Y_pred = reg.predict(X_test_pca)

        print(f"  Prediction: {Y_pred.shape}")

        mse = np.mean((Y_pred - Y_test) ** 2, axis=1)

        image_mse[test_idx] = mse
        image_seen[test_idx] = True

    if not np.all(image_seen):
        raise ValueError("Some images were never included in a test fold.")

    return image_mse

# ============================================================
# LOAD METADATA
# ============================================================

metadata_path = os.path.join(
    "PATH",
    "PATH",
    "PATH"
)

metadata = pd.read_csv(metadata_path)

print(f"Metadata shape: {metadata.shape}")

# ============================================================
# PROCESS ALL SUBJECTS AND ROIs
# ============================================================

for subject_number in range(1, 9):

    subject = f"subj{subject_number:02d}"

    print("\n" + "=" * 70)
    print(f"SUBJECT: {subject}")
    print("=" * 70)

    subject_roi_path = os.path.join(
        roi_root,
        subject
    )

    roi_files = glob.glob(
        os.path.join(subject_roi_path, "*.npy")
    )

    roi_files.sort()

    print(f"Number of ROIs: {len(roi_files)}")

    subject_results = {}

    for roi_path in roi_files:

        roi_name = os.path.basename(roi_path).replace(".npy", "")

        print("\n" + "-" * 70)
        print(f"ROI: {roi_name}")
        print("-" * 70)

        # ----------------------------------------------------
        # LOAD fMRI DATA
        # ----------------------------------------------------

        Y = np.load(roi_path)

        print(f"fMRI shape: {Y.shape}")

        if Y.shape[0] != len(metadata):
            raise ValueError(f"{roi_name}: fMRI images ({Y.shape[0]}) do not match metadata ({len(metadata)}).")

        # ----------------------------------------------------
        # STORE MSE FOR BOTH MODELS
        # ----------------------------------------------------

        model_results = {}

        # ====================================================
        # RUN THE TWO SELECTED MODELS
        # ====================================================

        for model_name in [MODEL1, MODEL2]:

            model_info = MODEL_CONFIG[model_name]
            feat_path = model_info["feat_path"]

            # ------------------------------------------------
            # FIND BEST LAYER FOR THIS MODEL × SUBJECT × ROI
            # ------------------------------------------------

            match = best_layers[
                (best_layers["Model"] == model_name) &
                (best_layers["Subject"] == subject) &
                (best_layers["ROI"] == roi_name)
            ]

            if len(match) == 0:
                raise ValueError(f"No best layer found for {model_name}, {subject}, {roi_name}")

            layer_id = match.iloc[0]["Best_Layer"]

            print("\n" + "=" * 50)
            print(f"{model_name} - {subject} - {roi_name} - {layer_id}")
            print("=" * 50)

            # ------------------------------------------------
            # LOAD FEATURE FILES
            # ------------------------------------------------

            feat_files = get_feature_files(feat_path)

            print(f"Feature files: {len(feat_files)}")

            if len(feat_files) != len(Y):
                raise ValueError(f"{model_name}: feature files ({len(feat_files)}) do not match fMRI images ({len(Y)}).")

            # ------------------------------------------------
            # LOAD BEST LAYER
            # ------------------------------------------------

            X = load_layer_features(feat_files, layer_id)

            print(f"Raw features: {X.shape}")

            # ------------------------------------------------
            # RUN IMAGEWISE RIDGE
            # ------------------------------------------------

            mse = run_imagewise_ridge(
                X,
                Y,
                n_folds=n_folds,
                n_components=n_components,
                alpha=alpha,
                random_state=random_state
            )

            model_results[model_name] = mse

            print(f"Final MSE: {mse.shape}")

        # ====================================================
        # CREATE COMPARISON DATAFRAME
        # ====================================================

        result_df = pd.DataFrame({
            "feature_index": np.arange(len(Y)),
            "nsd_id": metadata["nsd_id"].values,
            f"{MODEL1}_mse": model_results[MODEL1],
            f"{MODEL2}_mse": model_results[MODEL2]
        })

        # Lower MSE = better prediction

        result_df[f"{MODEL1}_rank"] = result_df[f"{MODEL1}_mse"].rank(method="min", ascending=True)
        result_df[f"{MODEL2}_rank"] = result_df[f"{MODEL2}_mse"].rank(method="min", ascending=True)

        # Positive = MODEL1 ranks better
        # Negative = MODEL2 ranks better

        result_df["rank_difference"] = result_df[f"{MODEL2}_rank"] - result_df[f"{MODEL1}_rank"]

        result_df["absolute_rank_difference"] = result_df["rank_difference"].abs()

        # ====================================================
        # SPEARMAN RANK CORRELATION
        # ====================================================

        rho, p_value = spearmanr(
            result_df[f"{MODEL1}_rank"],
            result_df[f"{MODEL2}_rank"]
        )

        print("\n" + "=" * 70)
        print("RANK COMPARISON")
        print("=" * 70)

        print(f"{MODEL1} vs {MODEL2}")
        print(f"Spearman rho: {rho:.4f}")
        print(f"p-value: {p_value:.4e}")

        # ====================================================
        # SAVE RESULTS
        # ====================================================

        subject_output = os.path.join(
            output_root,
            "results",
            subject
        )

        os.makedirs(subject_output, exist_ok=True)

        result_path = os.path.join(
            subject_output,
            f"{roi_name}_imagewise_results.csv"
        )

        result_df.to_csv(result_path, index=False)

        print(f"Saved: {result_path}")

        # ====================================================
        # SAVE TOP 20 FOR EACH MODEL
        # ====================================================

        model1_top20 = result_df.sort_values(
            f"{MODEL1}_mse"
        ).head(20)

        model2_top20 = result_df.sort_values(
            f"{MODEL2}_mse"
        ).head(20)

        disagreement_top20 = result_df.sort_values(
            "absolute_rank_difference",
            ascending=False
        ).head(20)

        model1_top20.to_csv(
            os.path.join(
                subject_output,
                f"{roi_name}_{MODEL1}_top20.csv"
            ),
            index=False
        )

        model2_top20.to_csv(
            os.path.join(
                subject_output,
                f"{roi_name}_{MODEL2}_top20.csv"
            ),
            index=False
        )

        disagreement_top20.to_csv(
            os.path.join(
                subject_output,
                f"{roi_name}_disagreement_top20.csv"
            ),
            index=False
        )

        # ====================================================
        # SAVE SUMMARY ROW
        # ====================================================

        subject_results[roi_name] = {
            "Spearman_rho": rho,
            "p_value": p_value,
            f"{MODEL1}_mean_MSE": result_df[f"{MODEL1}_mse"].mean(),
            f"{MODEL2}_mean_MSE": result_df[f"{MODEL2}_mse"].mean(),
            f"{MODEL1}_median_MSE": result_df[f"{MODEL1}_mse"].median(),
            f"{MODEL2}_median_MSE": result_df[f"{MODEL2}_mse"].median()
        }

    # ========================================================
    # SAVE SUBJECT SUMMARY
    # ========================================================

    summary_df = pd.DataFrame(subject_results).T
    summary_df.index.name = "ROI"

    summary_path = os.path.join(
        output_root,
        "results",
        subject,
        f"{subject}_summary.csv"
    )

    summary_df.to_csv(summary_path)

    print(f"\nSubject summary saved: {summary_path}")

# ============================================================
# DONE
# ============================================================

print("\n" + "=" * 70)
print("ALL SUBJECTS AND ROIs FINISHED")
print("=" * 70)

print(f"Comparison: {MODEL1} vs {MODEL2}")
print(f"Results saved in: {output_root}")

## visualizing

Calculate Spearman rank correlation per subject × ROI. It takes the already-generated image-wise results: and, for every:  
subject 1–8  
ROI in ["FFA-1", "EBA", "VWFA-1", "PPA"]  
calculates: rho, p = spearmanr(rank1, rank2)

In [ ]:
import os
import pandas as pd
from scipy.stats import spearmanr
########
RESULTS_ROOT = "PATH"

MODEL1 = "VJEPA"
MODEL2 = "LeVJEPA"
###########
ROIS = ["V1v","V1d","V2v","V2d","FFA-1", "EBA", "VWFA-1", "PPA"]

def calculate_spearman(model1, model2, rois, results_root):
    results = []

    for subject in range(1, 9):
        subj = f"subj{subject:02d}"

        for roi in rois:
            filename = f"{roi}_{subj}_imagewise_results.csv"
            path = os.path.join(results_root, subj, filename)

            if not os.path.exists(path):
                print(f"WARNING: File not found: {path}")
                continue

            df = pd.read_csv(path)

            rank1 = df[f"{model1}_rank"]
            rank2 = df[f"{model2}_rank"]

            rho, p = spearmanr(rank1, rank2)

            results.append({"Subject": subj, "ROI": roi, "Model1": model1, "Model2": model2, "Spearman_Rho": rho, "P_value": p})

    return pd.DataFrame(results)

spearman_results = calculate_spearman(MODEL1, MODEL2, ROIS, RESULTS_ROOT)

display(spearman_results)

In [ ]:
# aggregates the 8 subjects together for each ROI. And calc mean and SD of the 8 for each ROI
summary = spearman_results.groupby("ROI")["Spearman_Rho"].agg(["mean", "std"]).reset_index()

summary = summary.rename(columns={"mean": "Mean_Spearman", "std": "SD_Spearman"})

display(summary)

In [ ]:
output_dir = os.path.join(RESULTS_ROOT, f"{MODEL1}-{MODEL2}_spearman")
os.makedirs(output_dir, exist_ok=True)

pair_name = f"{MODEL1}_{MODEL2}"

subject_results_path = os.path.join(output_dir, f"{pair_name}_spearman_by_subject_roi.csv")
summary_path = os.path.join(output_dir, f"{pair_name}_spearman_summary.csv")

spearman_results.to_csv(subject_results_path, index=False)
summary.to_csv(summary_path, index=False)

print(f"Saved subject-level results to: {subject_results_path}")
print(f"Saved summary to: {summary_path}")

### violin plot 

In [ ]:
import plotly.express as px

fig = px.violin(
    spearman_results,
    x="ROI",
    y="Spearman_Rho",
    box=True,
    points="all",
    hover_data=["Subject"],
    title=f"{MODEL1} vs {MODEL2}: Image-Rank Correlation Across Subjects"
)

fig.update_layout(
    xaxis_title="ROI",
    yaxis_title="Spearman ρ",
)

fig.show(renderer="notebook")

### scatter plot

In [ ]:
# rendering. not nececarry anymore
#import plotly.io as pio
#pio.renderers.default = "notebook"
#print("Renderer set to:", pio.renderers.default)

In [ ]:
# NSD ID → image file mapping
import os
import glob
import re

image_root = "PATH"

image_files = glob.glob(os.path.join(image_root, "*.png"))

nsd_to_image = {}

for image_path in image_files:
    filename = os.path.basename(image_path)
    match = re.search("PATH", filename)
    if match:
        nsd_id = int(match.group(1))
        nsd_to_image[nsd_id] = image_path

print(f"Images mapped: {len(nsd_to_image)}")
print(f"Example: NSD 2950 -> {nsd_to_image[2950]}")

In [ ]:
import pandas as pd

path = "PATH"

df = pd.read_csv(path)

#print(df.shape)
#print(df.columns.tolist())

scatter plot. no img display

In [ ]:

import plotly.graph_objects as go
import os
import webbrowser

fig = go.Figure(data=[go.Scatter(x=df["MAE_rank"], y=df["LEJEPA_rank"], mode="markers", customdata=df["nsd_id"], marker=dict(size=6), hovertemplate="NSD ID: %{customdata}<br>MAE rank: %{x}<br>LEJEPA rank: %{y}<extra></extra>")])

fig.update_layout(title="MAE vs LEJEPA Image Ranking — subj01, FFA", xaxis_title="MAE Image Rank", yaxis_title="LEJEPA Image Rank", width=900, height=800)

html_path = os.path.abspath("MAE_LEJEPA_FFA_subj01_clickable.html")

fig.write_html(html_path, include_plotlyjs=True, auto_open=False)

webbrowser.open("file://" + html_path)

print(f"Saved: {html_path}")

### scatter with img display

In [ ]:
import json
import base64
import os
import webbrowser

# ============================================================
# PREPARE DATA
# ============================================================

points = []

for _, row in df.iterrows():

    nsd_id = int(row["nsd_id"])

    image_path = nsd_to_image[nsd_id]

    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    points.append({
        "nsd_id": nsd_id,
        "MAE_rank": float(row["MAE_rank"]),
        "LEJEPA_rank": float(row["LEJEPA_rank"]),
        "image": "data:image/png;base64," + encoded
    })


# ============================================================
# CONVERT TO JSON
# ============================================================

points_json = json.dumps(points)


# ============================================================
# HTML
# ============================================================

html = f"""
<!DOCTYPE html>
<html>

<head>

<meta charset="utf-8">

<title>MAE vs LEJEPA Image Ranking</title>

<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

<style>

body {{
    font-family: Arial, sans-serif;
    text-align: center;
}}

#plot {{
    width: 900px;
    height: 800px;
    margin: auto;
}}

#image-container {{
    margin-top: 20px;
}}

#selected-image {{
    max-width: 500px;
    max-height: 500px;
}}

</style>

</head>


<body>

<h2>MAE vs LEJEPA Image Ranking — subj01, FFA</h2>

<div id="plot"></div>

<div id="image-container">

<h3 id="image-title">
Click a point to display its image
</h3>

<img id="selected-image">

</div>


<script>

const points = {points_json};


// ------------------------------------------------------------
// CREATE PLOT
// ------------------------------------------------------------

const trace = {{

    x: points.map(p => p.MAE_rank),

    y: points.map(p => p.LEJEPA_rank),

    mode: "markers",

    type: "scatter",

    customdata: points.map(p => p.nsd_id),

    marker: {{
        size: 7
    }},

    hovertemplate:
        "NSD ID: %{{customdata}}" +
        "<br>MAE rank: %{{x}}" +
        "<br>LEJEPA rank: %{{y}}" +
        "<extra></extra>"
}};


const layout = {{

    title: "MAE vs LEJEPA Image Ranking — subj01, FFA",

    xaxis: {{
        title: "MAE Image Rank"
    }},

    yaxis: {{
        title: "LEJEPA Image Rank"
    }},

    width: 900,

    height: 800

}};


// ------------------------------------------------------------
// DRAW PLOT
// ------------------------------------------------------------

Plotly.newPlot(
    "plot",
    [trace],
    layout
).then(function() {{

    const plot = document.getElementById("plot");


    // --------------------------------------------------------
    // CLICK EVENT
    // --------------------------------------------------------

    plot.on("plotly_click", function(data) {{

        const point = data.points[0];

        const nsd_id = point.customdata;

        const selected = points.find(
            p => p.nsd_id === nsd_id
        );


        if (selected) {{

            document.getElementById(
                "selected-image"
            ).src = selected.image;


            document.getElementById(
                "image-title"
            ).innerText =
                "NSD ID: " + nsd_id;

        }}

    }});

}});

</script>

</body>

</html>
"""


# ============================================================
# SAVE
# ============================================================

html_path = os.path.abspath(
    "MAE_LEJEPA_FFA_subj01_clickable.html"
)

with open(
    html_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(html)


# ============================================================
# OPEN
# ============================================================

webbrowser.open(
    "file://" + html_path
)

print("Saved:")
print(html_path)

### automate all plots

In [ ]:
# automate for all rois and subjs
import os
import json
import base64
import pandas as pd


# ============================================================
# SETTINGS
# ============================================================

results_root = (
    "PATH"
    "PATH"
    "PATH"
    "PATH"
)

output_root = os.path.join(
    results_root,
    "scatter plots"
)

os.makedirs(
    output_root,
    exist_ok=True
)


# ============================================================
# SUBJECTS
# ============================================================

subjects = range(1, 9)


# ============================================================
# ROIS
# ============================================================

rois = [
    "V1v",
    "V1d",
    "V2v",
    "V2d",
    "V3v",
    "V3d",
    "hV4",
    "OPA",
    "PPA",
    "RSC",
    "OFA",
    "FFA-1",
    "EBA",
    "OWFA",
    "VWFA-1",
]


# ============================================================
# GENERATE ALL PLOTS
# ============================================================

total = 0
successful = 0
missing = 0
errors = 0


for subject in subjects:

    subj = f"subj{subject:02d}"

    print("\n" + "=" * 70)
    print(f"SUBJECT: {subj}")
    print("=" * 70)


    # --------------------------------------------------------
    # Subject input folder
    # --------------------------------------------------------

    subject_folder = os.path.join(
        results_root,
        subj
    )


    # --------------------------------------------------------
    # Subject output folder
    # --------------------------------------------------------

    subject_output_folder = os.path.join(
        output_root,
        subj
    )

    os.makedirs(
        subject_output_folder,
        exist_ok=True
    )


    # ========================================================
    # LOOP OVER ROIs
    # ========================================================

    for roi in rois:

        total += 1

        print(f"\nProcessing {subj} — {roi}")


        # ----------------------------------------------------
        # CSV path
        # ----------------------------------------------------

        csv_name = (
            f"{roi}_{subj}"
            f"_imagewise_results.csv"
        )

        csv_path = os.path.join(
            subject_folder,
            csv_name
        )


        # ----------------------------------------------------
        # Check CSV
        # ----------------------------------------------------

        if not os.path.exists(csv_path):

            print(
                f"  WARNING: CSV not found:\n"
                f"  {csv_path}"
            )

            missing += 1
            continue


        try:

            # ------------------------------------------------
            # Load data
            # ------------------------------------------------

            df = pd.read_csv(csv_path)


            # ------------------------------------------------
            # Check required columns
            # ------------------------------------------------

            required_columns = [
                "MAE_rank",
                "LEJEPA_rank",
                "nsd_id"
            ]

            missing_columns = [
                col
                for col in required_columns
                if col not in df.columns
            ]

            if missing_columns:

                print(
                    f"  ERROR: Missing columns: "
                    f"{missing_columns}"
                )

                errors += 1
                continue


            # =================================================
            # PREPARE POINTS
            # =================================================

            points = []


            for _, row in df.iterrows():

                nsd_id = int(
                    row["nsd_id"]
                )

                image_path = nsd_to_image[
                    nsd_id
                ]


                # ---------------------------------------------
                # Read image
                # ---------------------------------------------

                with open(
                    image_path,
                    "rb"
                ) as f:

                    encoded = base64.b64encode(
                        f.read()
                    ).decode("utf-8")


                points.append({

                    "nsd_id":
                        nsd_id,

                    "MAE_rank":
                        float(row["MAE_rank"]),

                    "LEJEPA_rank":
                        float(row["LEJEPA_rank"]),

                    "image":
                        "data:image/png;base64,"
                        + encoded
                })


            # =================================================
            # CONVERT TO JSON
            # =================================================

            points_json = json.dumps(
                points
            )


            # =================================================
            # HTML
            # =================================================

            html = f"""
<!DOCTYPE html>
<html>

<head>

<meta charset="utf-8">

<title>
MAE vs LEJEPA Image Ranking — {subj}, {roi}
</title>

<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

<style>

body {{
    font-family: Arial, sans-serif;
    text-align: center;
}}

#plot {{
    width: 900px;
    height: 800px;
    margin: auto;
}}

#image-container {{
    margin-top: 20px;
}}

#selected-image {{
    max-width: 500px;
    max-height: 500px;
}}

</style>

</head>


<body>

<h2>
MAE vs LEJEPA Image Ranking — {subj}, {roi}
</h2>

<div id="plot"></div>

<div id="image-container">

<h3 id="image-title">
Click a point to display its image
</h3>

<img
    id="selected-image"
>

</div>


<script>

const points = {points_json};


// ============================================================
// CREATE PLOT
// ============================================================

const trace = {{

    x: points.map(
        p => p.MAE_rank
    ),

    y: points.map(
        p => p.LEJEPA_rank
    ),

    mode: "markers",

    type: "scatter",

    customdata: points.map(
        p => p.nsd_id
    ),

    marker: {{
        size: 7
    }},

    hovertemplate:
        "NSD ID: %{{customdata}}" +
        "<br>MAE rank: %{{x}}" +
        "<br>LEJEPA rank: %{{y}}" +
        "<extra></extra>"
}};


const layout = {{

    title:
        "MAE vs LEJEPA Image Ranking — "
        + "{subj}, {roi}",

    xaxis: {{
        title: "MAE Image Rank"
    }},

    yaxis: {{
        title: "LEJEPA Image Rank"
    }},

    width: 900,

    height: 800

}};


// ============================================================
// DRAW PLOT
// ============================================================

Plotly.newPlot(
    "plot",
    [trace],
    layout
).then(function(){{

    const plot =
        document.getElementById("plot");


    // --------------------------------------------------------
    // CLICK EVENT
    // --------------------------------------------------------

    plot.on(
        "plotly_click",
        function(data){{

            const point =
                data.points[0];

            const nsd_id =
                point.customdata;

            const selected =
                points.find(
                    p => p.nsd_id === nsd_id
                );


            if (selected){{

                document.getElementById(
                    "selected-image"
                ).src =
                    selected.image;


                document.getElementById(
                    "image-title"
                ).innerText =
                    "NSD ID: "
                    + nsd_id;

            }}

        }}
    );

}});

</script>

</body>

</html>
"""


            # =================================================
            # SAVE HTML
            # =================================================

            html_filename = (
                f"MAE_LEJEPA_"
                f"{roi}_"
                f"{subj}_"
                f"clickable.html"
            )

            html_path = os.path.join(
                subject_output_folder,
                html_filename
            )


            with open(
                html_path,
                "w",
                encoding="utf-8"
            ) as f:

                f.write(html)


            print(
                f"  Saved: {html_filename}"
            )

            successful += 1


        except Exception as e:

            print(
                f"  ERROR processing "
                f"{subj} — {roi}:"
            )

            print(
                f"  {type(e).__name__}: {e}"
            )

            errors += 1


# ============================================================
# SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("COMPLETE")
print("=" * 70)

print(f"Total plots expected: {total}")
print(f"Successfully generated: {successful}")
print(f"Missing CSVs: {missing}")
print(f"Errors: {errors}")

print(
    f"\nPlots saved to:\n{output_root}"
)

## Brain activation

In [ ]:
activation = Y.mean(axis=1)

print("Mean:", activation.mean())
print("Std:", activation.std())
print("Min:", activation.min())
print("Max:", activation.max())
print("Median:", np.median(activation))

In [ ]:
print("Lowest 5:", np.sort(activation)[:5])
print("Highest 5:", np.sort(activation)[-5:])

Load the existing subj01 / FFA-1 MAE–LEJEPA results and verify that the 872 images are in the same order as the 872 rows of Y.

In [ ]:
import pandas as pd

results_path = "PATH"

df = pd.read_csv(results_path)

print("Results shape:", df.shape)
print("First 5 NSD IDs:", df["nsd_id"].head().tolist())

Compare the first 5 NSD IDs in df with the NSD IDs used by the ROI data. We can get the NSD IDs from the image filenames.

In [ ]:
import os
import re

image_dir = "PATH"

image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(".png")])

nsd_ids = [int(re.search("PATH", f)]

print("Number of NSD images:", len(nsd_ids))
print("First 5 NSD IDs:", nsd_ids[:5])

This gives every image its model-independent mean FFA-1 voxel activation.

In [ ]:
df["brain_activation"] = activation

print(df[["nsd_id", "brain_activation"]].head())
print("Activation values:", df["brain_activation"].shape)

see whether the brain activation and the model ranks already have an obvious relationship numerically, before we change the visualization.

In [ ]:
print(df[["MAE_rank", "LEJEPA_rank", "brain_activation"]].corr(method="spearman"))

create the activation value we will use for coloring, but normalize it only for visualization so the color scale is easier to interpret.

In [ ]:
from scipy.stats import zscore

df["activation_z"] = zscore(df["brain_activation"])

print(df[["nsd_id", "brain_activation", "activation_z"]].head())

### plot

In [ ]:
# ============================================================
# SELECT ANALYSIS
# ============================================================

SUBJECT = "subj01"
ROI = "EBA"
MODEL_X = "INFONCE"
MODEL_Y = "LeVLJEPA"


# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import json
import base64
import os
import webbrowser


# ============================================================
# PATHS
# ============================================================

results_dir = "PATH"
roi_dir = "PATH"
image_dir = "PATH"


# ============================================================
# LOAD RESULTS
# ============================================================

results_path = os.path.join(
    results_dir,
    SUBJECT,
    f"{ROI}_{SUBJECT}_imagewise_results.csv"
)

df_plot = pd.read_csv(results_path)


# ============================================================
# LOAD BRAIN ACTIVATION
# ============================================================

roi_path = os.path.join(
    roi_dir,
    SUBJECT,
    f"{ROI}_{SUBJECT}.npy"
)

Y = np.load(roi_path)

brain_activation = Y.mean(axis=1)

print("Brain activation statistics:")
print("Min:", brain_activation.min())
print("Max:", brain_activation.max())
print("Mean:", brain_activation.mean())
print("Std:", brain_activation.std())
print("Negative:", np.sum(brain_activation < 0))
print("Positive:", np.sum(brain_activation > 0))

df_plot["brain_activation"] = brain_activation


# ============================================================
# IMAGE MAPPING
# ============================================================

image_files = sorted(
    [
        f for f in os.listdir(image_dir)
        if f.endswith(".png")
    ]
)

nsd_to_image = {}

import re

for f in image_files:

    match = re.search("PATH", f)

    if match:
        nsd_id = int(match.group(1))
        nsd_to_image[nsd_id] = os.path.join(image_dir, f)


# ============================================================
# PREPARE DATA
# ============================================================

points_activation = []

for _, row in df_plot.iterrows():

    nsd_id = int(row["nsd_id"])

    image_path = nsd_to_image[nsd_id]

    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    points_activation.append({
        "nsd_id": nsd_id,
        "model_x_rank": float(row[f"{MODEL_X}_rank"]),
        "model_y_rank": float(row[f"{MODEL_Y}_rank"]),
        "brain_activation": float(row["brain_activation"]),
        "image": "data:image/png;base64," + encoded
    })


points_json = json.dumps(points_activation)


# ============================================================
# HTML
# ============================================================

html_activation = f"""
<!DOCTYPE html>
<html>

<head>

<meta charset="utf-8">

<title>{MODEL_X} vs {MODEL_Y} — Brain Activation</title>

<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

<style>

body {{
    font-family: Arial, sans-serif;
    text-align: center;
}}

#plot {{
    width: 900px;
    height: 800px;
    margin: auto;
}}

#image-container {{
    margin-top: 20px;
}}

#selected-image {{
    max-width: 500px;
    max-height: 500px;
}}

</style>

</head>


<body>

<h2>{MODEL_X} vs {MODEL_Y} Image Ranking — {SUBJECT}, {ROI}</h2>

<div id="plot"></div>

<div id="image-container">

<h3 id="image-title">
Click a point to display its image
</h3>

<img id="selected-image">

</div>


<script>

const points = {points_json};


// ------------------------------------------------------------
// CREATE PLOT
// ------------------------------------------------------------

const trace = {{

    x: points.map(p => p.model_x_rank),

    y: points.map(p => p.model_y_rank),

    mode: "markers",

    type: "scatter",

    customdata: points.map(p => [
        p.nsd_id,
        p.brain_activation
    ]),

    marker: {{

        size: 7,

        color: points.map(p => p.brain_activation),

        colorscale: "seismic",

        showscale: true,

        colorbar: {{
            title: "Brain activation"
        }}

    }},

    hovertemplate:
        "NSD ID: %{{customdata[0]}}" +
        "<br>{MODEL_X} rank: %{{x}}" +
        "<br>{MODEL_Y} rank: %{{y}}" +
        "<br>Brain activation: %{{customdata[1]:.3f}}" +
        "<extra></extra>"
}};


// ------------------------------------------------------------
// LAYOUT
// ------------------------------------------------------------

const layout = {{

    title: "{MODEL_X} vs {MODEL_Y} Image Ranking — {SUBJECT}, {ROI}",

    xaxis: {{
        title: "{MODEL_X} Image Rank"
    }},

    yaxis: {{
        title: "{MODEL_Y} Image Rank"
    }},

    width: 900,

    height: 800

}};


// ------------------------------------------------------------
// DRAW PLOT
// ------------------------------------------------------------

Plotly.newPlot(
    "plot",
    [trace],
    layout
).then(function(){{

    const plot = document.getElementById("plot");


    // --------------------------------------------------------
    // CLICK EVENT
    // --------------------------------------------------------

    plot.on("plotly_click", function(data){{

        const point = data.points[0];

        const nsd_id = point.customdata[0];

        const selected = points.find(
            p => p.nsd_id === nsd_id
        );


        if (selected){{

            document.getElementById(
                "selected-image"
            ).src = selected.image;


            document.getElementById(
                "image-title"
            ).innerText =
                "NSD ID: " + nsd_id +
                " | Brain activation: " +
                selected.brain_activation.toFixed(3);

        }}

    }});

}});

</script>

</body>

</html>
"""


# ============================================================
# SAVE
# ============================================================

html_activation_path = os.path.abspath(
    f"{MODEL_X}_{MODEL_Y}_{ROI}_{SUBJECT}_brain_activation.html"
)

with open(
    html_activation_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(html_activation)


# ============================================================
# OPEN
# ============================================================

webbrowser.open(
    "file://" + html_activation_path
)

print("Saved:")
print(html_activation_path)

In [ ]:
# inspecting old activations
import re
import json
import numpy as np

html_path = "PATH"

with open(html_path, ""PATH") as f:
    html = f.read()

match = re.search("PATH", html, re.DOTALL)

if not match:
    raise ValueError("Could not find the points data in the HTML.")

points = json.loads(match.group(1))
activation = np.array([p["brain_activation"] for p in points])

print("Number of points:", len(activation))
print("Min:", activation.min())
print("Max:", activation.max())
print("Mean:", activation.mean())
print("Std:", activation.std())
print("Negative:", np.sum(activation < 0))
print("Positive:", np.sum(activation > 0))

In [ ]:
# ============================================================
# SELECT ANALYSIS
# ============================================================

SUBJECT = "subj01"
ROI = "EBA"
MODEL_X = "INFONCE"
MODEL_Y = "LeVLJEPA"


# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import json
import base64
import os
import webbrowser


# ============================================================
# PATHS
# ============================================================

results_dir = "PATH"
roi_dir = "PATH"
image_dir = "PATH"


# ============================================================
# LOAD RESULTS
# ============================================================

results_path = os.path.join(
    results_dir,
    SUBJECT,
    f"{ROI}_{SUBJECT}_imagewise_results.csv"
)

df_plot = pd.read_csv(results_path)


# ============================================================
# LOAD BRAIN ACTIVATION
# ============================================================

roi_path = os.path.join(
    roi_dir,
    SUBJECT,
    f"{ROI}_{SUBJECT}.npy"
)

Y = np.load(roi_path)

brain_activation = Y.mean(axis=1)

print("Brain activation statistics:")
print("Min:", brain_activation.min())
print("Max:", brain_activation.max())
print("Mean:", brain_activation.mean())
print("Std:", brain_activation.std())
print("Negative:", np.sum(brain_activation < 0))
print("Positive:", np.sum(brain_activation > 0))

df_plot["brain_activation"] = brain_activation


# ============================================================
# IMAGE MAPPING
# ============================================================

image_files = sorted(
    [
        f for f in os.listdir(image_dir)
        if f.endswith(".png")
    ]
)

nsd_to_image = {}

import re

for f in image_files:

    match = re.search("PATH", f)

    if match:
        nsd_id = int(match.group(1))
        nsd_to_image[nsd_id] = os.path.join(image_dir, f)


# ============================================================
# PREPARE DATA
# ============================================================

points_activation = []

for _, row in df_plot.iterrows():

    nsd_id = int(row["nsd_id"])

    image_path = nsd_to_image[nsd_id]

    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    points_activation.append({
        "nsd_id": nsd_id,
        "model_x_rank": float(row[f"{MODEL_X}_rank"]),
        "model_y_rank": float(row[f"{MODEL_Y}_rank"]),
        "brain_activation": float(row["brain_activation"]),
        "image": "data:image/png;base64," + encoded
    })


points_json = json.dumps(points_activation)


# ============================================================
# HTML
# ============================================================

html_activation = f"""
<!DOCTYPE html>
<html>

<head>

<meta charset="utf-8">

<title>{MODEL_X} vs {MODEL_Y} — Brain Activation</title>

<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

<style>

body {{
    font-family: Arial, sans-serif;
    text-align: center;
}}

#plot {{
    width: 900px;
    height: 800px;
    margin: auto;
}}

#image-container {{
    margin-top: 20px;
}}

#selected-image {{
    max-width: 500px;
    max-height: 500px;
}}

</style>

</head>


<body>

<h2>{MODEL_X} vs {MODEL_Y} Image Ranking — {SUBJECT}, {ROI}</h2>

<div id="plot"></div>

<div id="image-container">

<h3 id="image-title">
Click a point to display its image
</h3>

<img id="selected-image">

</div>


<script>

const points = {points_json};


// ------------------------------------------------------------
// CREATE PLOT
// ------------------------------------------------------------

const trace = {{

    x: points.map(p => p.model_x_rank),

    y: points.map(p => p.model_y_rank),

    mode: "markers",

    type: "scatter",

    customdata: points.map(p => [
        p.nsd_id,
        p.brain_activation
    ]),

    marker: {{
        size: 7,
    
        color: points.map(p => p.brain_activation),
    
        colorscale: [
            [0.0, "blue"],
            [0.5, "white"],
            [1.0, "red"]
        ],
    
        cmin: -Math.max(
            ...points.map(p => Math.abs(p.brain_activation))
        ),
    
        cmax: Math.max(
            ...points.map(p => Math.abs(p.brain_activation))
        ),
    
        showscale: true,
    
        colorbar: {{
            title: "Brain activation"
        }}
    }},

    hovertemplate:
        "NSD ID: %{{customdata[0]}}" +
        "<br>{MODEL_X} rank: %{{x}}" +
        "<br>{MODEL_Y} rank: %{{y}}" +
        "<br>Brain activation: %{{customdata[1]:.3f}}" +
        "<extra></extra>"
}};


// ------------------------------------------------------------
// LAYOUT
// ------------------------------------------------------------

const layout = {{

    title: "{MODEL_X} vs {MODEL_Y} Image Ranking — {SUBJECT}, {ROI}",

    xaxis: {{
        title: "{MODEL_X} Image Rank"
    }},

    yaxis: {{
        title: "{MODEL_Y} Image Rank"
    }},

    width: 900,

    height: 800

}};


// ------------------------------------------------------------
// DRAW PLOT
// ------------------------------------------------------------

Plotly.newPlot(
    "plot",
    [trace],
    layout
).then(function(){{

    const plot = document.getElementById("plot");


    // --------------------------------------------------------
    // CLICK EVENT
    // --------------------------------------------------------

    plot.on("plotly_click", function(data){{

        const point = data.points[0];

        const nsd_id = point.customdata[0];

        const selected = points.find(
            p => p.nsd_id === nsd_id
        );


        if (selected){{

            document.getElementById(
                "selected-image"
            ).src = selected.image;


            document.getElementById(
                "image-title"
            ).innerText =
                "NSD ID: " + nsd_id +
                " | Brain activation: " +
                selected.brain_activation.toFixed(3);

        }}

    }});

}});

</script>

</body>

</html>
"""


# ============================================================
# SAVE
# ============================================================

html_activation_path = os.path.abspath(
    f"{MODEL_X}_{MODEL_Y}_{ROI}_{SUBJECT}_brain_activation.html"
)

with open(
    html_activation_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(html_activation)


# ============================================================
# OPEN
# ============================================================

webbrowser.open(
    "file://" + html_activation_path
)

print("Saved:")
print(html_activation_path)

### automate color plots for all rois, subjs

In [ ]:
# ============================================================
# SETTINGS
# ============================================================

MODEL_X = "VideoMAE" ######
MODEL_Y = "LeVJEPA" ######


# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import json
import base64
import os
import re


# ============================================================
# PATHS
# ============================================================

results_dir = (
    "PATH"
    "PATH"
    "PATH" ##############
    "PATH"
)

roi_dir = (
    "PATH" 
    "PATH"
)

image_dir = (
    "PATH"
    "PATH"
    "PATH"
)


# ============================================================
# OUTPUT
# ============================================================

output_dir = os.path.join(
    results_dir,
    "scatter plots with color",
    "brain activation"
)

os.makedirs(
    output_dir,
    exist_ok=True
)


# ============================================================
# SUBJECTS
# ============================================================

subjects = range(1, 9)


# ============================================================
# ROIS
# ============================================================

rois = [
    "V1v",
    "V1d",
    "V2v",
    "V2d",
    "V3v",
    "V3d",
    "hV4",
    "OPA",
    "PPA",
    "RSC",
    "OFA",
    "FFA-1",
    "EBA",
    "OWFA",
    "VWFA-1",
]


# ============================================================
# IMAGE MAPPING
# ============================================================

print("Creating NSD image mapping...")

image_files = sorted(
    [
        f
        for f in os.listdir(image_dir)
        if f.endswith(".png")
    ]
)

nsd_to_image = {}

for f in image_files:

    match = re.search(
        "PATH",
        f
    )

    if match:

        nsd_id = int(
            match.group(1)
        )

        nsd_to_image[nsd_id] = os.path.join(
            image_dir,
            f
        )

print(
    f"Found {len(nsd_to_image)} NSD images."
)


# ============================================================
# GENERATE ALL PLOTS
# ============================================================

total = 0
successful = 0
missing = 0
errors = 0


for subject_number in subjects:

    SUBJECT = f"subj{subject_number:02d}"

    print("\n" + "=" * 70)
    print(f"SUBJECT: {SUBJECT}")
    print("=" * 70)


    # ========================================================
    # OUTPUT SUBJECT FOLDER
    # ========================================================

    subject_output_dir = os.path.join(
        output_dir,
        SUBJECT
    )

    os.makedirs(
        subject_output_dir,
        exist_ok=True
    )


    # ========================================================
    # LOOP OVER ROIS
    # ========================================================

    for ROI in rois:

        total += 1

        print(
            f"\nProcessing {SUBJECT} — {ROI}"
        )


        try:

            # =================================================
            # LOAD IMAGEWISE RESULTS
            # =================================================

            results_path = os.path.join(
                results_dir,
                SUBJECT,
                f"{ROI}_{SUBJECT}_imagewise_results.csv"
            )


            if not os.path.exists(results_path):

                print(
                    f"  WARNING: Results file not found:"
                    f"\n  {results_path}"
                )

                missing += 1
                continue


            df_plot = pd.read_csv(
                results_path
            )


            # =================================================
            # LOAD BRAIN ACTIVATION
            # =================================================

            roi_path = os.path.join(
                roi_dir,
                SUBJECT,
                f"{ROI}_{SUBJECT}.npy"
            )


            if not os.path.exists(roi_path):

                print(
                    f"  WARNING: ROI file not found:"
                    f"\n  {roi_path}"
                )

                missing += 1
                continue


            Y = np.load(
                roi_path
            )


            # -------------------------------------------------
            # Mean activation across voxels
            # -------------------------------------------------

            brain_activation = Y.mean(
                axis=1
            )


            # -------------------------------------------------
            # Check length
            # -------------------------------------------------

            if len(brain_activation) != len(df_plot):

                raise ValueError(
                    f"Number of brain activation values "
                    f"({len(brain_activation)}) does not match "
                    f"number of rows in CSV "
                    f"({len(df_plot)})."
                )


            df_plot[
                "brain_activation"
            ] = brain_activation


            # =================================================
            # PREPARE DATA
            # =================================================

            points_activation = []


            for _, row in df_plot.iterrows():

                nsd_id = int(
                    row["nsd_id"]
                )


                # ------------------------------------------------
                # Check image mapping
                # ------------------------------------------------

                if nsd_id not in nsd_to_image:

                    raise KeyError(
                        f"NSD ID {nsd_id} "
                        f"not found in image mapping."
                    )


                image_path = nsd_to_image[
                    nsd_id
                ]


                # ------------------------------------------------
                # Encode image
                # ------------------------------------------------

                with open(
                    image_path,
                    "rb"
                ) as f:

                    encoded = (
                        base64.b64encode(
                            f.read()
                        )
                        .decode("utf-8")
                    )


                points_activation.append({

                    "nsd_id":
                        nsd_id,

                    "model_x_rank":
                        float(
                            row[
                                f"{MODEL_X}_rank"
                            ]
                        ),

                    "model_y_rank":
                        float(
                            row[
                                f"{MODEL_Y}_rank"
                            ]
                        ),

                    "brain_activation":
                        float(
                            row[
                                "brain_activation"
                            ]
                        ),

                    "image":
                        "data:image/png;base64,"
                        + encoded
                })


            # =================================================
            # CONVERT TO JSON
            # =================================================

            points_json = json.dumps(
                points_activation
            )


            # =================================================
            # HTML
            # =================================================

            html_activation = f"""
<!DOCTYPE html>
<html>

<head>

<meta charset="utf-8">

<title>
{MODEL_X} vs {MODEL_Y} — Brain Activation — {SUBJECT}, {ROI}
</title>

<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

<style>

body {{
    font-family: Arial, sans-serif;
    text-align: center;
}}

#plot {{
    width: 900px;
    height: 800px;
    margin: auto;
}}

#image-container {{
    margin-top: 20px;
}}

#selected-image {{
    max-width: 500px;
    max-height: 500px;
}}

</style>

</head>


<body>

<h2>
{MODEL_X} vs {MODEL_Y} Image Ranking —
{SUBJECT}, {ROI}
</h2>

<div id="plot"></div>

<div id="image-container">

<h3 id="image-title">
Click a point to display its image
</h3>

<img id="selected-image">

</div>


<script>

const points = {points_json};


// ============================================================
// CREATE PLOT
// ============================================================

const trace = {{

    x: points.map(
        p => p.model_x_rank
    ),

    y: points.map(
        p => p.model_y_rank
    ),

    mode: "markers",

    type: "scatter",

    customdata: points.map(
        p => [
            p.nsd_id,
            p.brain_activation
        ]
    ),

    marker: {{

        size: 7,

        color: points.map(
            p => p.brain_activation
        ),

        colorscale: "seismic",

        showscale: true,

        colorbar: {{
            title: "Brain activation"
        }}

    }},

    hovertemplate:
        "NSD ID: %{{customdata[0]}}" +
        "<br>{MODEL_X} rank: %{{x}}" +
        "<br>{MODEL_Y} rank: %{{y}}" +
        "<br>Brain activation: %{{customdata[1]:.3f}}" +
        "<extra></extra>"
}};


// ============================================================
// LAYOUT
// ============================================================

const layout = {{

    title:
        "{MODEL_X} vs {MODEL_Y} Image Ranking — "
        + "{SUBJECT}, {ROI}",

    xaxis: {{
        title: "{MODEL_X} Image Rank"
    }},

    yaxis: {{
        title: "{MODEL_Y} Image Rank"
    }},

    width: 900,

    height: 800

}};


// ============================================================
// DRAW PLOT
// ============================================================

Plotly.newPlot(
    "plot",
    [trace],
    layout
).then(function(){{

    const plot =
        document.getElementById("plot");


    // --------------------------------------------------------
    // CLICK EVENT
    // --------------------------------------------------------

    plot.on(
        "plotly_click",
        function(data){{

            const point =
                data.points[0];

            const nsd_id =
                point.customdata[0];

            const selected =
                points.find(
                    p => p.nsd_id === nsd_id
                );


            if (selected){{

                document.getElementById(
                    "selected-image"
                ).src =
                    selected.image;


                document.getElementById(
                    "image-title"
                ).innerText =
                    "NSD ID: "
                    + nsd_id
                    + " | Brain activation: "
                    + selected.brain_activation.toFixed(3);

            }}

        }}
    );

}});

</script>

</body>

</html>
"""


            # =================================================
            # SAVE
            # =================================================

            html_filename = (
                f"{MODEL_X}_{MODEL_Y}_"
                f"{ROI}_{SUBJECT}_"
                f"brain_activation.html"
            )


            html_path = os.path.join(
                subject_output_dir,
                html_filename
            )


            with open(
                html_path,
                "w",
                encoding="utf-8"
            ) as f:

                f.write(
                    html_activation
                )


            print(
                f"  Saved: {html_filename}"
            )

            successful += 1


        except Exception as e:

            print(
                f"  ERROR: "
                f"{type(e).__name__}: {e}"
            )

            errors += 1


# ============================================================
# SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("COMPLETE")
print("=" * 70)

print(
    f"Total plots expected: {total}"
)

print(
    f"Successfully generated: {successful}"
)

print(
    f"Missing files: {missing}"
)

print(
    f"Errors: {errors}"
)

print(
    f"\nPlots saved to:\n{output_dir}"
)

### normal scatter plot for everything + downloadf

In [ ]:
# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = "PATH"

ROI_DIR = (
    "PATH"
    "PATH"
)

IMAGE_DIR = (
    "PATH"
    "PATH"
    "PATH"
)


# ============================================================
# IMPORTS
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


# ============================================================
# MODEL PAIRS
# ============================================================

MODEL_PAIRS = {
    "DINOv1-LeJEPA-B": {
        "folde"PATH",
        "model_x": "DINOv1",
        "model_y": "LeJEPA-B",
    },

    "DINOv2-LeVJEPA": {
        "folde"PATH",
        "model_x": "DINOv2",
        "model_y": "LeVJEPA",
    },

    "DINOv2-VJEPA": {
        "folde"PATH",
        "model_x": "DINOv2",
        "model_y": "VJEPA",
    },

    "IBOT-LeJEPA": {
        "folde"PATH",
        "model_x": "IBOT",
        "model_y": "LeJEPA",
    },

    "INFONCE-LeVLJEPA": {
        "folde"PATH",
        "model_x": "INFONCE",
        "model_y": "LeVLJEPA",
    },

    "MAE-LeJEPA": {
        "folde"PATH",
        "model_x": "MAE",
        "model_y": "LEJEPA",
    },

    "OPENCLIP-LeVLJEPA": {
        "folde"PATH",
        "model_x": "OPENCLIP",
        "model_y": "LeVLJEPA",
    },

    "VideoMAE-LeVJEPA": {
        "folde"PATH",
        "model_x": "VideoMAE",
        "model_y": "LeVJEPA",
    },

    "VideoMAE-VJEPA": {
        "folde"PATH",
        "model_x": "VideoMAE",
        "model_y": "VJEPA",
    },

    "VJEPA-LeVJEPA": {
        "folde"PATH",
        "model_x": "VJEPA",
        "model_y": "LeVJEPA",
    },
}


# ============================================================
# SUBJECTS
# ============================================================

SUBJECTS = [
    f"subj{i:02d}"
    for i in range(1, 9)
]


# ============================================================
# ROIS
# ============================================================

ROIS = [
    "V1v",
    "V1d",
    "V2v",
    "V2d",
    "V3v",
    "V3d",
    "hV4",
    "OPA",
    "PPA",
    "RSC",
    "OFA",
    "FFA-1",
    "EBA",
    "OWFA",
    "VWFA-1",
]


# ============================================================
# OUTPUT FOLDER
# ============================================================

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "scatter plots - brain activation"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# IMAGE MAPPING
# ============================================================

print("Creating NSD image mapping...")

image_files = sorted(
    [
        f
        for f in os.listdir(IMAGE_DIR)
        if f.endswith(".png")
    ]
)

nsd_to_image = {}

for f in image_files:

    match = re.search(
        "PATH",
        f
    )

    if match:

        nsd_id = int(
            match.group(1)
        )

        nsd_to_image[nsd_id] = os.path.join(
            IMAGE_DIR,
            f
        )


print(
    f"Found {len(nsd_to_image)} NSD images."
)


# ============================================================
# COUNTERS
# ============================================================

total = 0
successful = 0
missing = 0
errors = 0


# ============================================================
# LOOP OVER MODEL PAIRS
# ============================================================

for pair_name, pair_info in MODEL_PAIRS.items():

    MODEL_X = pair_info["model_x"]
    MODEL_Y = pair_info["model_y"]
    pair_folder = pair_info["folder"]


    print("\n")
    print("=" * 80)
    print(f"MODEL PAIR: {pair_name}")
    print(f"{MODEL_X} vs {MODEL_Y}")
    print("=" * 80)


    # --------------------------------------------------------
    # Results folder for this model pair
    # --------------------------------------------------------

    results_dir = os.path.join(
        BASE_DIR,
        pair_folder,
        "results"
    )


    # --------------------------------------------------------
    # Output folder for this model pair
    # --------------------------------------------------------

    pair_output_dir = os.path.join(
        OUTPUT_DIR,
        pair_name
    )

    os.makedirs(
        pair_output_dir,
        exist_ok=True
    )


    # ========================================================
    # LOOP OVER SUBJECTS
    # ========================================================

    for SUBJECT in SUBJECTS:

        print(
            f"\nProcessing {pair_name} — {SUBJECT}"
        )


        # ----------------------------------------------------
        # Subject output folder
        # ----------------------------------------------------

        subject_output_dir = os.path.join(
            pair_output_dir,
            SUBJECT
        )

        os.makedirs(
            subject_output_dir,
            exist_ok=True
        )


        # ====================================================
        # LOOP OVER ROIS
        # ====================================================

        for ROI in ROIS:

            total += 1

            print(
                f"  {ROI}"
            )


            try:

                # =================================================
                # LOAD IMAGEWISE RESULTS
                # =================================================

                results_path = os.path.join(
                    results_dir,
                    SUBJECT,
                    f"{ROI}_{SUBJECT}_imagewise_results.csv"
                )


                if not os.path.exists(results_path):

                    print(
                        f"    WARNING: Results file not found"
                    )

                    missing += 1

                    continue


                df_plot = pd.read_csv(
                    results_path
                )


                # =================================================
                # CHECK MODEL RANK COLUMNS
                # =================================================

                x_rank_column = f"{MODEL_X}_rank"
                y_rank_column = f"{MODEL_Y}_rank"


                if x_rank_column not in df_plot.columns:

                    raise KeyError(
                        f"Missing column: {x_rank_column}"
                    )


                if y_rank_column not in df_plot.columns:

                    raise KeyError(
                        f"Missing column: {y_rank_column}"
                    )


                # =================================================
                # LOAD BRAIN ACTIVATION
                # =================================================

                roi_path = os.path.join(
                    ROI_DIR,
                    SUBJECT,
                    f"{ROI}_combined_{SUBJECT}.npy"
                )


                if not os.path.exists(roi_path):

                    print(
                        f"    WARNING: ROI file not found"
                    )

                    missing += 1

                    continue


                Y = np.load(
                    roi_path
                )


                # -------------------------------------------------
                # Mean activation across voxels
                # -------------------------------------------------

                brain_activation = Y.mean(
                    axis=1
                )


                # -------------------------------------------------
                # Check length
                # -------------------------------------------------

                if len(brain_activation) != len(df_plot):

                    raise ValueError(
                        f"Number of brain activation values "
                        f"({len(brain_activation)}) does not match "
                        f"number of rows in CSV "
                        f"({len(df_plot)})."
                    )


                df_plot[
                    "brain_activation"
                ] = brain_activation


                # =================================================
                # PREPARE DATA
                # =================================================

                x = df_plot[
                    x_rank_column
                ].to_numpy()

                y = df_plot[
                    y_rank_column
                ].to_numpy()

                activation = df_plot[
                    "brain_activation"
                ].to_numpy()


                # =================================================
                # CREATE FIGURE
                # =================================================

                fig, ax = plt.subplots(
                    figsize=(9, 8)
                )


                # =================================================
                # SEISMIC COLOR NORMALIZATION
                # =================================================

                # Center the colormap at zero so that:
                # negative activation -> blue
                # zero               -> white
                # positive activation -> red

                max_abs_activation = np.max(
                    np.abs(activation)
                )

                norm = TwoSlopeNorm(
                    vmin=-max_abs_activation,
                    vcenter=0,
                    vmax=max_abs_activation
                )


                # =================================================
                # SCATTER
                # =================================================

                scatter = ax.scatter(
                    x,
                    y,
                    c=activation,
                    cmap="seismic",
                    norm=norm,
                    s=28,
                    alpha=0.75,
                    edgecolors="none"
                )


                # =================================================
                # DIAGONAL
                # =================================================

                ax.plot(
                    [1, 872],
                    [1, 872],
                    linestyle="--",
                    linewidth=1,
                    alpha=0.5
                )


                # =================================================
                # COLORBAR
                # =================================================

                colorbar = fig.colorbar(
                    scatter,
                    ax=ax
                )

                colorbar.set_label(
                    "Average brain activation"
                )


                # =================================================
                # AXES
                # =================================================

                ax.set_xlabel(
                    f"{MODEL_X} image rank"
                )

                ax.set_ylabel(
                    f"{MODEL_Y} image rank"
                )


                ax.set_xlim(
                    1,
                    872
                )

                ax.set_ylim(
                    1,
                    872
                )


                # Keep the same visual scale
                # for x and y because both are ranks.

                ax.set_aspect(
                    "equal",
                    adjustable="box"
                )


                # =================================================
                # TITLE
                # =================================================

                ax.set_title(
                    f"{MODEL_X} vs {MODEL_Y} Image Ranking\n"
                    f"{SUBJECT} — {ROI}"
                )


                # =================================================
                # GRID
                # =================================================

                ax.grid(
                    alpha=0.2
                )


                # =================================================
                # SAVE
                # =================================================

                output_path = os.path.join(
                    subject_output_dir,
                    f"{ROI}.png"
                )


                plt.tight_layout()

                plt.savefig(
                    output_path,
                    dpi=200,
                    bbox_inches="tight"
                )

                plt.close(
                    fig
                )


                print(
                    f"    Saved: {output_path}"
                )

                successful += 1


            except Exception as e:

                print(
                    f"    ERROR: "
                    f"{type(e).__name__}: {e}"
                )

                errors += 1


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 80)
print("COMPLETE")
print("=" * 80)

print(
    f"Total plots expected: {total}"
)

print(
    f"Successfully generated: {successful}"
)

print(
    f"Missing files: {missing}"
)

print(
    f"Errors: {errors}"
)

print(
    f"\nPlots saved to:\n{OUTPUT_DIR}"
)

### top 20

In [ ]:
top20 = df.nlargest(20, "brain_activation")[["nsd_id", "brain_activation", "MAE_rank", "LEJEPA_rank"]]

print(top20.to_string(index=False))

In [ ]:
# It produces a clean 3 × 10 figure with the top 10 images for FFA-1, EBA, and PPA, ranked by average brain activation, and saves it as a high-resolution PNG.
# ============================================================
# SELECT ANALYSIS
# ============================================================

SUBJECT = "subj01"
ROIS = ["FFA-1", "EBA", "PPA"]


# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import re


# ============================================================
# PATHS
# ============================================================

roi_dir = "PATH"
image_dir = "PATH"


# ============================================================
# IMAGE MAPPING
# ============================================================

image_files = sorted(
    [
        f for f in os.listdir(image_dir)
        if f.endswith(".png")
    ]
)

nsd_to_image = {}

for f in image_files:

    match = re.search("PATH", f)

    if match:
        nsd_id = int(match.group(1))
        nsd_to_image[nsd_id] = os.path.join(image_dir, f)


# ============================================================
# GET TOP 10 IMAGES FOR EACH ROI
# ============================================================

top_images = {}

for ROI in ROIS:

    # --------------------------------------------------------
    # Load ROI brain activity
    # --------------------------------------------------------

    roi_path = os.path.join(
        roi_dir,
        SUBJECT,
        f"{ROI}_{SUBJECT}.npy"
    )

    Y = np.load(roi_path)

    # --------------------------------------------------------
    # Average activation across all voxels
    # --------------------------------------------------------

    brain_activation = Y.mean(axis=1)

    # --------------------------------------------------------
    # NSD IDs in the same image order
    # --------------------------------------------------------

    nsd_ids = metadata["nsd_id"].values

    # --------------------------------------------------------
    # Find 10 highest-activation images
    # --------------------------------------------------------

    top_indices = np.argsort(brain_activation)[::-1][:10]

    top_images[ROI] = []

    for i in top_indices:

        nsd_id = int(nsd_ids[i])

        top_images[ROI].append({
            "nsd_id": nsd_id,
            "activation": float(brain_activation[i]),
            "image_path": nsd_to_image[nsd_id]
        })


# ============================================================
# PRINT TOP 10 TABLES
# ============================================================

for ROI in ROIS:

    print("\n" + "=" * 55)
    print(f"{ROI} — Top 10 Images")
    print("=" * 55)

    print(f"{'Rank':<6}{'NSD ID':<10}{'Activation':<12}")

    for rank, item in enumerate(top_images[ROI], start=1):

        print(
            f"{rank:<6}"
            f"{item['nsd_id']:<10}"
            f"{item['activation']:.4f}"
        )


# ============================================================
# CREATE FIGURE
# ============================================================

fig, axes = plt.subplots(
    nrows=3,
    ncols=10,
    figsize=(20, 7)
)


# ============================================================
# ADD IMAGES
# ============================================================

for row, ROI in enumerate(ROIS):

    for col, item in enumerate(top_images[ROI]):

        image = Image.open(item["image_path"])

        axes[row, col].imshow(image)

        axes[row, col].axis("off")

        axes[row, col].set_title(
            f"#{col + 1}\n"
            f"NSD {item['nsd_id']}\n"
            f"Activation: {item['activation']:.3f}",
            fontsize=8
        )

    # ROI label on the left

    axes[row, 0].text(
        -0.55,
        0.5,
        ROI,
        transform=axes[row, 0].transAxes,
        fontsize=13,
        fontweight="bold",
        va="center",
        ha="right"
    )


# ============================================================
# FIGURE TITLE
# ============================================================

fig.suptitle(
    "Top 10 Images by Average Brain Activation — Subject 1",
    fontsize=16,
    fontweight="bold"
)


# ============================================================
# LAYOUT
# ============================================================

plt.tight_layout(
    rect=[0.04, 0, 1, 0.94]
)


# ============================================================
# SAVE
# ============================================================

output_path = os.path.abspath(
    "subj01_top10_brain_activation_FFA1_EBA_PPA.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)


# ============================================================
# DISPLAY
# ============================================================

plt.show()

print("\nSaved to:")
print(output_path)


### top 10 for every model

In [ ]:
# ============================================================
# SUBJECT 1 — TOP 10 IMAGES FOR EVERY MODEL AND ROI
# ============================================================

SUBJECT = "subj01"


# ============================================================
# IMPORTS
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


# ============================================================
# PATHS
# ============================================================

base_dir = "PATH"

image_dir = "PATH"


# ============================================================
# MODEL PAIRS
# ============================================================

model_pairs = {
    "DINOv1": "PATH",
    "LeJEPA-B": "PATH",

    "DINOv2": "PATH",
    "VJEPA": "PATH",

    "LeVJEPA": "DINOv2-LeVJEPA best layer",

    "IBOT": "IBOT-LeJEPA best layer",

    "InfoNCE": "INFONCE-LeVLJEPA best layer",
    "LeVLJEPA": "INFONCE-LeVLJEPA best layer",

    "MAE": "MAE-LeJEPA best layer",

    "OpenCLIP": "OPENCLIP-LeVLJEPA best layer",

    "VideoMAE": "VideoMAE-VJEPA best layer",

    "LeVJepa": "VideoMAE-LeVJEPA best layer"
}


# ============================================================
# ROIS
# ============================================================

ROIS = [
    "EBA",
    "FFA-1",
    "OFA",
    "OPA",
    "OWFA",
    "PPA",
    "RSC",
    "V1d",
    "V1v",
    "V2d",
    "V2v",
    "V3d",
    "V3v",
    "VWFA-1",
    "hV4"
]


# ============================================================
# IMAGE MAPPING
# ============================================================

image_files = sorted(
    [
        f for f in os.listdir(image_dir)
        if f.endswith(".png")
    ]
)

nsd_to_image = {}

for f in image_files:

    match = re.search("PATH", f)

    if match:

        nsd_id = int(match.group(1))

        nsd_to_image[nsd_id] = os.path.join(
            image_dir,
            f
        )


# ============================================================
# OUTPUT FOLDER
# ============================================================

output_dir = os.path.join(
    base_dir,
    "top10_ranked_images_subject1"
)

os.makedirs(
    output_dir,
    exist_ok=True
)


# ============================================================
# PROCESS EACH MODEL
# ============================================================

for model, pair_folder in model_pairs.items():

    print("\n" + "=" * 70)
    print(f"Processing {model}")
    print("=" * 70)


    # --------------------------------------------------------
    # Create figure
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        nrows=len(ROIS),
        ncols=10,
        figsize=(20, 30)
    )


    # --------------------------------------------------------
    # Process every ROI
    # --------------------------------------------------------

    for row, ROI in enumerate(ROIS):

        csv_path = os.path.join(
            base_dir,
            pair_folder,
            "results",
            SUBJECT,
            f"{ROI}_{SUBJECT}_{model}_top20.csv"
        )


        # ----------------------------------------------------
        # Check file
        # ----------------------------------------------------

        if not os.path.exists(csv_path):

            print(
                f"WARNING: Missing file for "
                f"{model} / {ROI}"
            )

            for col in range(10):
                axes[row, col].axis("off")

            continue


        # ----------------------------------------------------
        # Load ranking
        # ----------------------------------------------------

        df = pd.read_csv(csv_path)


        # ----------------------------------------------------
        # Find model rank column
        # ----------------------------------------------------

        rank_columns = [
            c for c in df.columns
            if c.lower() == f"{model.lower()}_rank"
        ]


        if not rank_columns:

            print(
                f"WARNING: No rank column for "
                f"{model} / {ROI}"
            )

            print(
                "Available columns:",
                df.columns.tolist()
            )

            for col in range(10):
                axes[row, col].axis("off")

            continue


        rank_column = rank_columns[0]


        # ----------------------------------------------------
        # Select 10 best-ranked images
        # ----------------------------------------------------

        top10 = (
            df
            .sort_values(rank_column)
            .head(10)
        )


        # ----------------------------------------------------
        # Display images
        # ----------------------------------------------------

        for col, (_, item) in enumerate(top10.iterrows()):

            nsd_id = int(item["nsd_id"])

            rank = float(item[rank_column])

            image_path = nsd_to_image.get(nsd_id)


            if image_path is None:

                axes[row, col].text(
                    0.5,
                    0.5,
                    f"NSD {nsd_id}\n"
                    "image not found",
                    ha="center",
                    va="center"
                )

                axes[row, col].axis("off")

                continue


            image = Image.open(image_path)

            axes[row, col].imshow(image)

            axes[row, col].axis("off")

            axes[row, col].set_title(
                f"#{col + 1}\n"
                f"NSD {nsd_id}\n"
                f"rank {rank:.0f}",
                fontsize=7
            )


        # ----------------------------------------------------
        # ROI label
        # ----------------------------------------------------

        axes[row, 0].text(
            -0.55,
            0.5,
            ROI,
            transform=axes[row, 0].transAxes,
            fontsize=11,
            fontweight="bold",
            va="center",
            ha="right"
        )


    # --------------------------------------------------------
    # Figure title
    # --------------------------------------------------------

    fig.suptitle(
        f"{model} — Subject 1\n"
        "Top 10 Images by ROI-Specific Image Ranking",
        fontsize=18,
        fontweight="bold"
    )


    # --------------------------------------------------------
    # Layout
    # --------------------------------------------------------

    plt.tight_layout(
        rect=[0.04, 0, 1, 0.96]
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    output_path = os.path.join(
        output_dir,
        f"{model}_subj01_top10_ROI_images.png"
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)


    print("Saved:")
    print(output_path)


print("\nAll Subject 1 figures created.")

### top 10 for Lejepa for all subjects

In [ ]:
# ============================================================
# LEJEPA — TOP 10 IMAGES ACROSS ALL SUBJECTS
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


# ============================================================
# PATHS
# ============================================================

base_dir = "PATH"

image_dir = "PATH"


# ============================================================
# MODEL / PAIR
# ============================================================

MODEL = "LeJEPA"

# LeJEPA rankings are available in the MAE-LeJEPA pair
pair_folder = "MAE-LeJEPA best layer"


# ============================================================
# SUBJECTS
# ============================================================

SUBJECTS = [
    "subj01",
    "subj02",
    "subj03",
    "subj04",
    "subj05",
    "subj06",
    "subj07",
    "subj08"
]


# ============================================================
# ROIS
# ============================================================

ROIS = [
    "EBA",
    "FFA-1",
    "OFA",
    "OPA",
    "OWFA",
    "PPA",
    "RSC",
    "V1d",
    "V1v",
    "V2d",
    "V2v",
    "V3d",
    "V3v",
    "VWFA-1",
    "hV4"
]


# ============================================================
# IMAGE MAPPING
# ============================================================

image_files = sorted(
    [
        f for f in os.listdir(image_dir)
        if f.endswith(".png")
    ]
)

nsd_to_image = {}

for f in image_files:

    match = re.search(
        "PATH",
        f
    )

    if match:

        nsd_id = int(match.group(1))

        nsd_to_image[nsd_id] = os.path.join(
            image_dir,
            f
        )


# ============================================================
# OUTPUT FOLDER
# ============================================================

output_dir = os.path.join(
    base_dir,
    "LeJEPA_top10_images_across_subjects"
)

os.makedirs(
    output_dir,
    exist_ok=True
)


# ============================================================
# PROCESS EACH ROI
# ============================================================

for ROI in ROIS:

    print("\n" + "=" * 70)
    print(f"Processing {ROI}")
    print("=" * 70)


    # --------------------------------------------------------
    # Create figure
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        nrows=8,
        ncols=10,
        figsize=(20, 14)
    )


    # --------------------------------------------------------
    # Process every subject
    # --------------------------------------------------------

    for row, SUBJECT in enumerate(SUBJECTS):

        csv_path = os.path.join(
            base_dir,
            pair_folder,
            "results",
            SUBJECT,
            f"{ROI}_SUBJECT}_{MODEL}_top20.csv"
        )


        # ----------------------------------------------------
        # Check file
        # ----------------------------------------------------

        if not os.path.exists(csv_path):

            print(
                f"WARNING: Missing file: "
                f"{SUBJECT} / {ROI}"
            )

            for col in range(10):
                axes[row, col].axis("off")

            continue


        # ----------------------------------------------------
        # Load ranking
        # ----------------------------------------------------

        df = pd.read_csv(csv_path)


        # ----------------------------------------------------
        # Find rank column
        # ----------------------------------------------------

        rank_columns = [
            c for c in df.columns
            if c.lower() == "lejepa_rank"
        ]


        if not rank_columns:

            print(
                f"WARNING: No LeJEPA rank column "
                f"for {SUBJECT} / {ROI}"
            )

            for col in range(10):
                axes[row, col].axis("off")

            continue


        rank_column = rank_columns[0]


        # ----------------------------------------------------
        # Select 10 best-ranked images
        # ----------------------------------------------------

        top10 = (
            df
            .sort_values(rank_column)
            .head(10)
        )


        # ----------------------------------------------------
        # Display images
        # ----------------------------------------------------

        for col, (_, item) in enumerate(
            top10.iterrows()
        ):

            nsd_id = int(item["nsd_id"])

            rank = float(
                item[rank_column]
            )

            image_path = nsd_to_image.get(
                nsd_id
            )


            if image_path is None:

                axes[row, col].text(
                    0.5,
                    0.5,
                    f"NSD {nsd_id}\n"
                    "image not found",
                    ha="center",
                    va="center"
                )

                axes[row, col].axis("off")

                continue


            image = Image.open(
                image_path
            )

            axes[row, col].imshow(
                image
            )

            axes[row, col].axis("off")

            axes[row, col].set_title(
                f"#{col + 1}\n"
                f"NSD {nsd_id}\n"
                f"rank {rank:.0f}",
                fontsize=7
            )


        # ----------------------------------------------------
        # Subject label
        # ----------------------------------------------------

        axes[row, 0].text(
            -0.55,
            0.5,
            SUBJECT,
            transform=axes[row, 0].transAxes,
            fontsize=11,
            fontweight="bold",
            va="center",
            ha="right"
        )


    # --------------------------------------------------------
    # Figure title
    # --------------------------------------------------------

    fig.suptitle(
        f"LeJEPA — {ROI}\n"
        "Top 10 Images by ROI-Specific Image Ranking",
        fontsize=18,
        fontweight="bold"
    )


    # --------------------------------------------------------
    # Layout
    # --------------------------------------------------------

    plt.tight_layout(
        rect=[0.04, 0, 1, 0.94]
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    output_path = os.path.join(
        output_dir,
        f"LeJEPA_{ROI}_top10_across_subjects.png"
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)


    print("Saved:")
    print(output_path)


print("\nAll 15 ROI figures created.")